In [2]:
from collections import OrderedDict
import torch

input_size = 5
hidden_dims = 10
output_size = 2

net = torch.nn.Sequential(
    OrderedDict(
        [
            ("layer1", torch.nn.Linear(input_size, hidden_dims)),
            ("layer2", torch.nn.Linear(hidden_dims, output_size))
        ]
    )
).requires_grad_(False)

In [3]:
from nnsight import NNsight

tiny_model = NNsight(net)

In [4]:
print(tiny_model)

Sequential(
  (layer1): Linear(in_features=5, out_features=10, bias=True)
  (layer2): Linear(in_features=10, out_features=2, bias=True)
)


In [5]:
input = torch.rand((1, input_size))

with tiny_model.trace(input) as tracer:
    pass

In [7]:
with tiny_model.trace(input) as tracer:
    output = tiny_model.output.save()
print(output)

tensor([[-0.5459, -0.0612]])


In [16]:
with tiny_model.trace(input) as tracer:
    l1_out = tiny_model.layer1.output.save()
    l2_in = tiny_model.layer2.input.save()
print(l1_out)
print(l2_in)

tensor([[-0.2001, -0.0656, -0.2127, -0.9332, -0.6389, -0.1356,  0.2791, -0.2149,
          0.7869, -0.3678]])
tensor([[-0.2001, -0.0656, -0.2127, -0.9332, -0.6389, -0.1356,  0.2791, -0.2149,
          0.7869, -0.3678]])


In [17]:
with tiny_model.trace(input):

    l1_output = tiny_model.layer1.output

    l1_amax = torch.argmax(l1_output, dim=1).save()

print(l1_amax[0])

tensor(8)


In [18]:
with tiny_model.trace(input):
    value = (tiny_model.layer1.output.sum() + tiny_model.layer2.output.sum()).save()
print(value)

tensor(-2.3098)


In [19]:
with tiny_model.trace(input):
    l1_output_before = tiny_model.layer1.output.clone().save()
    tiny_model.layer1.output[:, 0] = 0
    l1_output_after = tiny_model.layer1.output.save()

print(l1_output_before)
print(l1_output_after)

tensor([[-0.2001, -0.0656, -0.2127, -0.9332, -0.6389, -0.1356,  0.2791, -0.2149,
          0.7869, -0.3678]])
tensor([[ 0.0000, -0.0656, -0.2127, -0.9332, -0.6389, -0.1356,  0.2791, -0.2149,
          0.7869, -0.3678]])


In [20]:
with tiny_model.trace(input):
    # 1) access l1 & l2 outputs so trace knows these are intermediate values we care about
    l1_output = tiny_model.layer1.output

    # 2) make sure gradient flows back to l1 (it will pass by l2)
    l1_output.requires_grad = True
    l2_output = tiny_model.layer2.output

    # 3) access gradients within a backwards trace
    with tiny_model.output.sum().backward():
        # access .grad within backward context in REVERSE ORDER
        layer2_output_grad = l2_output.grad.save()
        layer1_output_grad = l1_output.grad.save()

print(f"{layer1_output_grad=}")
print(f"{layer2_output_grad=}")


layer1_output_grad=tensor([[-0.2126,  0.0071,  0.4248,  0.0972,  0.5141,  0.0524, -0.0155, -0.0738,
         -0.4012,  0.2890]])
layer2_output_grad=tensor([[1., 1.]])
